<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/MNV2_Entrenada_contra_Gini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenamiento

In [2]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json

import kagglehub
path4 = kagglehub.dataset_download("leonardocaravaggio/ge-images4")
path5 = kagglehub.dataset_download("leonardocaravaggio/ge-images5")

mkdir: cannot create directory ‘/root/.kaggle’: File exists


100%|██████████| 2.89G/2.89G [00:40<00:00, 76.2MB/s]

Extracting files...


100%|██████████| 895M/895M [00:14<00:00, 64.7MB/s]

Extracting files...


In [24]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
from tqdm import tqdm


# ==== PARÁMETROS ====
IMG_TYPE = "10K"  # Puede ser "1K", "5K", "10K", "15K"
BATCH_SIZE = 8
EPOCHS = 12
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHT_DECAY=0.005

# ==== CARGA ====
df_eph = pd.read_csv("Gini_EPH.csv")
df_ocde = pd.read_csv(path4 + "/Gini con latlon.csv")

# ==== NORMALIZAR COLUMNAS ====
# Renombrar columnas para que coincidan
df_eph = df_eph.rename(columns={"Nombre_Aglomerado": "Ciudad", "Gini_Hogares": "Gini"})
df_ocde["Gini"] = df_ocde["Gini"].str.replace(',', '.', regex=False).astype(float)

# ==== CONCATENAR ====
df_merged = pd.concat([df_eph[["Ciudad", "Gini"]], df_ocde[["Ciudad", "Gini"]]], ignore_index=True)


# ==== TRANSFORMACIONES ====
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def sanear_nombre_ciudad(nombre):
    return nombre.replace("/", ".").replace(":", "_").replace("'", "!")

# ==== DATASET PERSONALIZADO ====
import os
from PIL import Image

class GiniDataset(torch.utils.data.Dataset):
    def __init__(self, df, path1, path2, transform=None):
        self.df = df.copy()
        self.path1 = path1
        self.path2 = path2
        self.transform = transform
        self.sufijos = [" - 1K.png", " - 5K.png", " - 10K.png", " - 15K.png"]

        # Expandimos el DataFrame para que cada ciudad tenga 4 filas (una por imagen)
        self.expanded_rows = []
        for _, row in self.df.iterrows():
            for suf in self.sufijos:
                self.expanded_rows.append({
                    "Ciudad": row["Ciudad"],
                    "Gini": row["Gini"],
                    "Sufijo": suf
                })

    def sanear_nombre_ciudad(self, nombre):
        nombre = nombre.replace("/", ".").replace(":", "_").replace("'", "!").replace("\\", "").strip()
        return nombre

    def __getitem__(self, idx):
        row = self.expanded_rows[idx]
        ciudad = row["Ciudad"]
        sufijo = row["Sufijo"]
        gini = row["Gini"]

        nombre_archivo = self.sanear_nombre_ciudad(ciudad)
        img_path = None

        for base_path in [self.path1, self.path2]:
            path = os.path.join(base_path, f"{nombre_archivo}{sufijo}")
            if os.path.exists(path):
                img_path = path
                break

        if img_path is None:
            print(f"⚠️ No se encontró la imagen para {ciudad} - {sufijo}")
            image = Image.new("RGB", (512, 512), (0, 0, 0))  # Imagen negra placeholder
        else:
            image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(gini, dtype=torch.float32)
        return image, label

    def __len__(self):
        return len(self.expanded_rows)



# ==== CARGA DE DATOS ====
train_df, val_df = train_test_split(df_merged, test_size=0.2, random_state=42)
train_dataset = GiniDataset(train_df, path4, path5, transform)
val_dataset = GiniDataset(val_df, path4, path5, transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ==== MODELO ====
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
mobilenet_features = nn.Sequential(*list(mobilenet.features))  # todos los bloques
for param in mobilenet_features[:].parameters():
    param.requires_grad = True

model = nn.Sequential(
    mobilenet_features,                     # output: (batch_size, 1280, H, W)
    nn.AdaptiveAvgPool2d((1,1)),            # output: (batch_size, 1280, 1, 1)
    nn.Flatten(),                           # output: (batch_size, 1280)
    nn.Linear(1280, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(DEVICE)


# ==== OPTIMIZADOR Y PÉRDIDA ====
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# ==== ENTRENAMIENTO ====
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Train Loss: {running_loss/len(train_loader):.4f}")

# ==== VALIDACIÓN ====
model.eval()
preds_val, trues_val = [], []
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs).cpu().numpy().flatten()
        preds_val.extend(outputs)
        trues_val.extend(targets.numpy())

r_val, p_val = pearsonr(preds_val, trues_val)

# Evaluar también en entrenamiento para referencia
preds_train, trues_train = [], []
with torch.no_grad():
    for inputs, targets in train_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs).cpu().numpy().flatten()
        preds_train.extend(outputs)
        trues_train.extend(targets.numpy())

r_train, p_train = pearsonr(preds_train, trues_train)

print(f"\n📊 Pearson Train: {r_train:.3f} | p-value Train: {p_train:.5f}")
print(f"📊 Pearson Val: {r_val:.3f} | p-value Val: {p_val:.5f}")

Epoch 1/12: 100%|██████████| 58/58 [09:00<00:00,  9.32s/it]


Train Loss: 0.0081


Epoch 2/12: 100%|██████████| 58/58 [08:57<00:00,  9.27s/it]


Train Loss: 0.0022


Epoch 3/12: 100%|██████████| 58/58 [09:01<00:00,  9.34s/it]


...

Train Loss: 0.0011


Epoch 11/12: 100%|██████████| 58/58 [09:16<00:00,  9.60s/it]


Train Loss: 0.0010


Epoch 12/12: 100%|██████████| 58/58 [09:07<00:00,  9.44s/it]


Train Loss: 0.0010

📊 Pearson Train: 0.839 | p-value Train: 0.00000
📊 Pearson Val: 0.400 | p-value Val: 0.00001


# Validación

In [19]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
path1 = kagglehub.dataset_download("leonardocaravaggio/ge-images")
path2 = kagglehub.dataset_download("leonardocaravaggio/ge-images2")
path3 = kagglehub.dataset_download("leonardocaravaggio/ge-images3")

mkdir: cannot create directory ‘/root/.kaggle’: File exists


100%|██████████| 13.3G/13.3G [06:00<00:00, 39.6MB/s]

Extracting files...


100%|██████████| 13.9G/13.9G [06:19<00:00, 39.4MB/s]

Extracting files...


100%|██████████| 336M/336M [00:07<00:00, 44.4MB/s]

Extracting files...


In [20]:
import os
import shutil

# Crear una carpeta de destino
dest_folder = "imagenes"
os.makedirs(dest_folder, exist_ok=True)

# Función para copiar imágenes a una sola carpeta
def mover_imagenes(origen, destino):
    for root, _, files in os.walk(origen):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                shutil.move(os.path.join(root, file), os.path.join(destino, file))

# Copiar imágenes de ambos datasets al mismo folder
mover_imagenes(path1, dest_folder)
mover_imagenes(path2, dest_folder)

print(f"Imágenes combinadas en la carpeta: {dest_folder}")

Imágenes combinadas en la carpeta: imagenes


In [25]:
from PIL import Image
import torch
import os

# Asegurate de que tu modelo esté en modo evaluación
model.eval()

# Dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms (usá el mismo que en entrenamiento)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # o el tamaño que usaste
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet means
                         std=[0.229, 0.224, 0.225])
])

# Lista de ciudades
lista_ciudades = [
    "Desierto",  "Amazonas", "Oceano", "Santiago de Cuba", "Curitiba", "El Alto",
    "Montevideo", "Lo Barnechea", "Los Cedros", "La Cava", "Rocinha", "Retiro"
]

# Evaluar cada ciudad
for nombre in lista_ciudades:
    nombre_archivo = nombre.replace("/", ".").replace(":", "_").replace("'", "!")
    ruta_completa = os.path.join(path3, nombre_archivo)

    print(f"\nCiudad: {nombre}")

    for escala in ["1K", "5K", "10K", "15K"]:
        path_img = f"{ruta_completa} - {escala}.png"

        try:
            image = Image.open(path_img).convert("RGB")
            image = transform(image).unsqueeze(0).to(device)

            with torch.no_grad():
                pred = model(image).item()

            print(f"  {escala}: {pred:.2f}")

        except FileNotFoundError:
            print(f"  {escala}: ❌ Imagen no encontrada ({path_img})")
        except Exception as e:
            print(f"  {escala}: ⚠️ Error al procesar ({e})")



Ciudad: Desierto
  1K: 0.33
  5K: 0.30
  10K: 0.33
  15K: 0.31

Ciudad: Amazonas
  1K: 0.32
  5K: 0.31
  10K: 0.29
  15K: 0.29

Ciudad: Oceano
  1K: 0.44
  5K: 0.44
  10K: 0.45
  15K: 0.44

Ciudad: Santiago de Cuba
  1K: 0.37
  5K: 0.39
  10K: 0.34
  15K: 0.30

Ciudad: Curitiba
  1K: 0.31
  5K: 0.35
  10K: 0.40
  15K: 0.28

Ciudad: El Alto
  1K: 0.39
  5K: 0.35
  10K: 0.40
  15K: 0.39

Ciudad: Montevideo
  1K: 0.34
  5K: 0.40
  10K: 0.33
  15K: 0.29

Ciudad: Lo Barnechea
  1K: 0.38
  5K: 0.34
  10K: 0.39
  15K: 0.39

Ciudad: Los Cedros
  1K: 0.33
  5K: 0.29
  10K: 0.28
  15K: 0.33

Ciudad: La Cava
  1K: 0.39
  5K: 0.40
  10K: 0.42
  15K: 0.30

Ciudad: Rocinha
  1K: 0.33
  5K: 0.34
  10K: 0.34
  15K: 0.33

Ciudad: Retiro
  1K: 0.40
  5K: 0.38
  10K: 0.34
  15K: 0.38


# Inferencia

In [32]:
import pandas as pd
import numpy as np
import os
from PIL import Image
from tqdm import tqdm
import torch

# Cargar base
ciudades = pd.read_csv("base.csv")

# Asegurar columnas
for km in ["1km", "5km", "10km", "15km"]:
    ciudades[f'Desigualdad_{km}'] = np.nan

# Procesamiento por ciudad
for idx, row in tqdm(ciudades.iterrows(), total=len(ciudades)):
    nombre = row['City']
    nombre_archivo = nombre.replace("/", ".").replace(":", "_").replace("'", "!")
    ruta_completa = os.path.join('imagenes', nombre_archivo)

    for escala, km_col in zip(["1K", "5K", "10K", "15K"], ["1km", "5km", "10km", "15km"]):
        path_img = f"{ruta_completa} - {escala}.png"

        try:
            image = Image.open(path_img).convert("RGB")
            image = transform(image).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                pred = model(image).item()

            ciudades.at[idx, f'Desigualdad_{km_col}'] = pred

        except FileNotFoundError:
            ciudades.at[idx, f'Desigualdad_{km_col}'] = np.nan  # ya es nan, pero explícito
        except Exception as e:
            print(f"{nombre} {escala}: ⚠️ Error al procesar: {e}")


100%|██████████| 1095/1095 [16:22<00:00,  1.11it/s]


In [33]:
from scipy.stats import pearsonr

# Eliminar pares con NaN
x = ciudades["Desigualdad_10km"]
y = ciudades["P1ST"]
mask = x.notna() & y.notna()

# Calcular correlación de Pearson y p-value
r, p = pearsonr(x[mask], y[mask])

print(f"Coeficiente de Pearson: {r:.3f}")
print(f"Valor p: {p:.5f}")

Coeficiente de Pearson: 0.068
Valor p: 0.02504
